# Days 1–15: Full Statistics Toolkit Applied to the Titanic Dataset

This notebook contains every piece of code from Day 1 through Day 15, applied to one real
dataset, step by step, in simple beginner-friendly Python. No shortcuts, no clever one-liners —
every step is written out and commented so it's easy to follow and re-read later.


## Objective

- Clean a messy real-world dataset (Titanic passenger data) from scratch
- Apply every statistics concept from Day 1 to Day 14, one at a time, on this same dataset
- Finish with a full Day 15 EDA pass that ties everything together
- Save a final, fully cleaned CSV file at the end


## Imports

In [1]:
# Libraries we need for this whole notebook

import pandas as pd            # for working with tables of data
import numpy as np             # for handling missing values (NaN)
from scipy import stats        # for statistical tests (t-test, chi-square, etc.)
from scipy.stats import skew   # for measuring skewness


## Step 0: Cleaning the Data

Before any statistics, we clean the raw file. Real data is almost never perfect — this dataset
has extra spaces, inconsistent capitalization, a currency symbol hiding inside a number column,
and a fake "missing" text marker. We fix these one at a time, checking the result after each fix.


In [2]:
# Load the raw file
df = pd.read_csv('titanic_messy.csv')

# Look at the first few rows
print(df.head())


   PassengerId  Survived  Pclass  \
0          404         0       3   
1          863         1       1   
2          591         0       3   
3          243         0       2   
4          438         1       2   

                                                Name     Sex   Age  SibSp  \
0                     Hakkarainen, Mr. Pekka Pietari    male  28.0      1   
1  Swift, Mrs. Frederick Joel (Margaret Welles Ba...  Female  48.0      0   
2                               Rintamaki, Mr. Matti    male  35.0      0   
3                    Coleridge, Mr. Reginald Charles    MALE  29.0      0   
4              Richards, Mrs. Sidney (Emily Hocking)  FEMALE  24.0      2   

   Parch             Ticket     Fare Cabin     Embarked  
0      0   STON/O2. 3101279    15.85   NaN            S  
1      0              17466  25.9292   D17  Southampton  
2      0  STON/O 2. 3101273    7.125   NaN            S  
3      0        W./C. 14263     10.5   NaN            s  
4      3              29106   

In [3]:
# Check which columns have missing values
missing_values = df.isna().sum()
print(missing_values)


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            180
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          682
Embarked         2
dtype: int64


In [4]:
# Check the Sex column for inconsistent spelling and spacing
print(df['Sex'].unique())


<StringArray>
['male', 'Female', 'MALE', 'FEMALE', ' male', ' female ', 'Male', 'female']
Length: 8, dtype: str


In [5]:
# Fix 1: remove extra spaces from Sex
df['Sex'] = df['Sex'].str.strip()

# Fix 2: make every value lowercase
df['Sex'] = df['Sex'].str.lower()

# Check the result
print(df['Sex'].unique())


<StringArray>
['male', 'female']
Length: 2, dtype: str


In [6]:
# Fix 3: some Fare values were typed with a dollar sign, like "$13.00"
# This makes the whole column text instead of numbers

df['Fare'] = df['Fare'].astype(str)             # make sure every value is text first
df['Fare'] = df['Fare'].str.replace('$', '')    # remove the dollar sign
df['Fare'] = df['Fare'].astype(float)           # convert the text into real numbers

# Check the result
print(df['Fare'].dtype)


float64


In [7]:
# Fix 4: the Cabin column uses the text "N/A" to mean missing
# Replace that text with a real missing value marker (NaN)

df['Cabin'] = df['Cabin'].replace('N/A', np.nan)

print(df['Cabin'].isna().sum())


682


In [8]:
# Fix 5: clean the Embarked column the same way we cleaned Sex

df['Embarked'] = df['Embarked'].str.strip()
df['Embarked'] = df['Embarked'].str.lower()

print(df['Embarked'].unique())


<StringArray>
['s', 'southampton', 'c', 'queenstown', 'cherbourg', 'q', nan]
Length: 7, dtype: str


In [9]:
# Fix 6: Embarked still has letters and full city names mixed together
# Map them all to one standard letter

embarked_map = {
    's': 'S', 'southampton': 'S',
    'c': 'C', 'cherbourg': 'C',
    'q': 'Q', 'queenstown': 'Q'
}

df['Embarked'] = df['Embarked'].map(embarked_map)

print(df['Embarked'].unique())


<StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str


In [10]:
# Fix 7: remove stray whitespace from Name

df['Name'] = df['Name'].str.strip()

print(df['Name'].head())


0                       Hakkarainen, Mr. Pekka Pietari
1    Swift, Mrs. Frederick Joel (Margaret Welles Ba...
2                                 Rintamaki, Mr. Matti
3                      Coleridge, Mr. Reginald Charles
4                Richards, Mrs. Sidney (Emily Hocking)
Name: Name, dtype: str


Data is now clean. Every day from here on uses this same cleaned `df`.

## Day 1 — Why Statistics?

**Simple idea:** instead of reading every row one by one, we squeeze the data into one number
that tells us something useful right away.


In [11]:
# Overall survival rate: the percentage of passengers who survived

survival_rate = df['Survived'].mean()
print(survival_rate)


0.3852097130242826


**Why this matters:** one line of code replaced manually counting through 906 rows. That's
the whole point of Day 1 — statistics compresses raw data into a decision-useful number.


## Day 2 — Central Tendency (Mean, Median, Mode)

**Simple idea:** mean is the average, median is the middle value, mode is the most common value.
We need all three because one alone can be misleading.


In [12]:
# Drop missing ages first, we can't average empty cells
age = df['Age'].dropna()
fare = df['Fare']

# Mean, median, and mode for Age
age_mean = age.mean()
age_median = age.median()
age_mode = age.mode()[0]

print("Age mean:", age_mean)
print("Age median:", age_median)
print("Age mode:", age_mode)


Age mean: 30.25643250688705
Age median: 28.0
Age mode: 24.0


In [13]:
# Mean, median, and mode for Fare
fare_mean = fare.mean()
fare_median = fare.median()
fare_mode = fare.mode()[0]

print("Fare mean:", fare_mean)
print("Fare median:", fare_median)
print("Fare mode:", fare_mode)


Fare mean: 32.02798675496689
Fare median: 14.4542
Fare mode: 8.05


**Why this matters:** if `fare_mean` is a lot bigger than `fare_median`, that tells us a few
expensive tickets are pulling the average up. Median is the more honest "typical value" for Fare
in that case.


## Day 3 — Variance & Standard Deviation

**Simple idea:** variance and standard deviation tell us how spread out the data is, not just
where the "middle" is.


In [14]:
# Variance and standard deviation for Age
age_var = age.var()
age_std = age.std()

print("Age variance:", age_var)
print("Age standard deviation:", age_std)


Age variance: 543.6283233933693
Age standard deviation: 23.31583846644528


In [15]:
# Variance and standard deviation for Fare
fare_var = fare.var()
fare_std = fare.std()

print("Fare variance:", fare_var)
print("Fare standard deviation:", fare_std)


Fare variance: 2436.7126183385867
Fare standard deviation: 49.3630693772033


In [16]:
# To compare spread fairly across two different units (years vs pounds),
# we divide standard deviation by the mean. This is called coefficient of variation.

cv_age = age_std / age_mean
cv_fare = fare_std / fare_mean

print("Coefficient of variation - Age:", cv_age)
print("Coefficient of variation - Fare:", cv_fare)


Coefficient of variation - Age: 0.7706076537985126
Coefficient of variation - Fare: 1.5412479640028605


**Why this matters:** the bigger the coefficient of variation, the more relatively spread
out that column is. This lets us compare Age and Fare fairly even though they're in different
units.


## Day 4 — Percentiles, IQR & Outliers

**Simple idea:** we find the middle 50% of the data (IQR), then flag anything far outside that
range as an outlier.


In [17]:
# Find the 25th and 75th percentile of Fare
q1 = fare.quantile(0.25)
q3 = fare.quantile(0.75)

# IQR is the width of the middle 50%
iqr = q3 - q1

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)


Q1: 7.92125
Q3: 31.0
IQR: 23.07875


In [18]:
# Anything beyond 1.5 times the IQR past Q1 or Q3 is considered an outlier

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)


Lower limit: -26.696875
Upper limit: 65.61812499999999


In [19]:
# Filter the rows where Fare is outside these limits
outliers = df[(df['Fare'] < lower_limit) | (df['Fare'] > upper_limit)]

print("Number of Fare outliers:", len(outliers))
outliers[['Name', 'Pclass', 'Fare']].sort_values('Fare', ascending=False).head(5)


Number of Fare outliers: 117


,Name,Pclass,Fare
166,"Cardeza, Mr. Thomas Drake Martinez",1,512.3292
621,"Lesurer, Mr. Gustave J",1,512.3292
525,"Ward, Miss. Anna",1,512.3292
380,"Fortune, Mr. Charles Alexander",1,263.0000
609,"Fortune, Miss. Mabel Helen",1,263.0000


**Why IQR and not standard deviation here:** Fare is skewed (a few very high values). The
standard deviation method assumes roughly symmetric data, so it can get fooled by skew. IQR is
based on ranks, not the mean, so it isn't distorted by skew.


## Day 5 — Distributions & Skewness

**Simple idea:** skewness is a number that tells us if the data leans to one side instead of
being symmetric.


In [20]:
# Skewness of Age and Fare

age_skew = skew(age)
fare_skew = skew(fare)

print("Age skewness:", age_skew)
print("Fare skewness:", fare_skew)


Age skewness: 5.479120311344645
Fare skewness: 4.805731644033497


**Why this matters:** a number close to 0 means roughly symmetric data. A large positive
number means a long tail stretching to the right — Fare has this because of a few very expensive
tickets, which matches what we saw in Day 2 and Day 4.


## Day 6 — Probability Basics

**Simple idea:** we compare survival chances between groups by filtering the data first, then
taking the average.


In [21]:
# Overall probability of survival
p_survived = df['Survived'].mean()

# Probability of being female
p_female = (df['Sex'] == 'female').mean()

print("P(Survived):", p_survived)
print("P(Female):", p_female)


P(Survived): 0.3852097130242826
P(Female): 0.3543046357615894


In [22]:
# Filter to only female passengers, then find their survival rate
female_only = df[df['Sex'] == 'female']
p_survived_given_female = female_only['Survived'].mean()

# Filter to only male passengers, then find their survival rate
male_only = df[df['Sex'] == 'male']
p_survived_given_male = male_only['Survived'].mean()

print("P(Survived | Female):", p_survived_given_female)
print("P(Survived | Male):", p_survived_given_male)


P(Survived | Female): 0.7445482866043613
P(Survived | Male): 0.18803418803418803


**Why this matters:** filtering the table first, then averaging Survived within just that
group, gives us the conditional probability. A big gap between these two numbers is a strong
early signal that Sex is linked to survival.


## Day 7 — Bayes' Theorem

**Simple idea:** Bayes lets us flip the question around — instead of "if female, survival
chance?" we ask "if survived, chance she was female?"


In [23]:
# Bayes' Theorem: P(Female | Survived) = P(Survived | Female) * P(Female) / P(Survived)

p_female_given_survived_bayes = (p_survived_given_female * p_female) / p_survived

print("P(Female | Survived) using Bayes formula:", p_female_given_survived_bayes)


P(Female | Survived) using Bayes formula: 0.6848137535816617


In [24]:
# Check this against the real data directly, by filtering survivors first

survivors_only = df[df['Survived'] == 1]
p_female_given_survived_direct = (survivors_only['Sex'] == 'female').mean()

print("P(Female | Survived) using direct filtering:", p_female_given_survived_direct)


P(Female | Survived) using direct filtering: 0.6848137535816619


**Why this matters:** both numbers should match closely. This proves Bayes' formula
actually works on real numbers, not just as an abstract theory.


## Day 8 — Correlation (Pearson vs Spearman)

**Simple idea:** Pearson checks straight-line relationships between two truly numeric columns.
Spearman checks rank-order relationships, which is better for ordinal data like Pclass.


In [25]:
# Pearson correlation between Age and Fare (both are truly numeric)

age_no_na = df['Age'].dropna()
fare_matching = df.loc[df['Age'].notna(), 'Fare']

pearson_corr, pearson_p_value = stats.pearsonr(age_no_na, fare_matching)

print("Pearson correlation (Age vs Fare):", pearson_corr)
print("p-value:", pearson_p_value)


Pearson correlation (Age vs Fare): 0.03049071472095923
p-value: 0.4120287408696311


In [26]:
# Spearman correlation between Pclass and Fare (Pclass is a rank, not a true number)

spearman_corr, spearman_p_value = stats.spearmanr(df['Pclass'], df['Fare'])

print("Spearman correlation (Pclass vs Fare):", spearman_corr)
print("p-value:", spearman_p_value)


Spearman correlation (Pclass vs Fare): -0.6884265729229466
p-value: 3.137604831575127e-128


In [27]:
# Spearman correlation between Pclass and Survived

spearman_corr2, spearman_p_value2 = stats.spearmanr(df['Pclass'], df['Survived'])

print("Spearman correlation (Pclass vs Survived):", spearman_corr2)
print("p-value:", spearman_p_value2)


Spearman correlation (Pclass vs Survived): -0.33550750292443565
p-value: 2.831340863847989e-25


**Why Spearman for Pclass:** Pclass (1st/2nd/3rd) is a rank, not a real numeric scale.
Spearman only cares about order, which matches what Pclass actually represents. The negative
correlation confirms a better class (lower number) is linked to higher Fare and higher survival.


## Day 9 — Hypothesis Testing Setup, and Day 10 — p-values / t-test

**Question:** do survivors and non-survivors have different average ages?

**H0 (null hypothesis):** the average age is the same for both groups.
**H1 (alternative hypothesis):** the average ages are different.

**Simple idea:** we split the data into two separate groups and compare their averages using a
t-test.


In [28]:
# Split Age into two groups: survivors and non-survivors

age_survived = df[df['Survived'] == 1]['Age'].dropna()
age_died = df[df['Survived'] == 0]['Age'].dropna()

print("Mean age (Survived):", age_survived.mean())
print("Mean age (Died):", age_died.mean())


Mean age (Survived): 27.99549152542373
Mean age (Died): 31.803944315545245


In [29]:
# Run an independent two-sample t-test
# equal_var=False means we don't assume both groups have the same spread (Welch's t-test)

t_stat, p_value = stats.ttest_ind(age_survived, age_died, equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)


t-statistic: -2.1714823214937677
p-value: 0.030263537647492393


In [30]:
# Check the result against our significance level (alpha = 0.05)

alpha = 0.05

if p_value < alpha:
    print("p < 0.05, so we reject H0. There is strong evidence of an age difference.")
else:
    print("p >= 0.05, so we fail to reject H0. No strong evidence of an age difference.")


p < 0.05, so we reject H0. There is strong evidence of an age difference.


**Why a t-test and not something else:** we're comparing the means of two separate,
unrelated groups. That's exactly what an independent two-sample t-test is designed for. Note the
careful wording — a p-value gives us "strong evidence," never "proof."


## Day 11 — Chi-Square Test

**Simple idea:** when both variables are categories (not numbers), we use chi-square instead of
a t-test.


In [31]:
# Build a count table: Sex vs Survived

contingency_sex = pd.crosstab(df['Sex'], df['Survived'])
print(contingency_sex)


Survived    0    1
Sex               
female     82  239
male      475  110


In [32]:
# Run the chi-square test on this table

chi2_sex, p_sex, dof_sex, expected_sex = stats.chi2_contingency(contingency_sex)

print("Chi-square statistic:", chi2_sex)
print("p-value:", p_sex)


Chi-square statistic: 268.7122971381176
p-value: 2.1655136161599326e-60


In [33]:
# Do the same for Pclass vs Survived

contingency_pclass = pd.crosstab(df['Pclass'], df['Survived'])
print(contingency_pclass)

chi2_pclass, p_pclass, dof_pclass, expected_pclass = stats.chi2_contingency(contingency_pclass)

print("Chi-square statistic:", chi2_pclass)
print("p-value:", p_pclass)


Survived    0    1
Pclass            
1          81  138
2          99   88
3         377  123
Chi-square statistic: 102.14573567883758
p-value: 6.596830454024382e-23


**Why this matters:** both p-values are far below 0.05, confirming Sex and Pclass are each
statistically linked to survival — matching what Day 6 and Day 8 already hinted at, but now
tested properly instead of just eyeballed.


## Day 12 — Central Limit Theorem (CLT)

**Simple idea:** even though Fare itself is skewed, if we take many small samples and average
each one, those averages end up looking roughly normal.


In [34]:
# Take 1000 samples of size 30 from Fare, and record each sample's mean

np.random.seed(42)
sample_means = []

for i in range(1000):
    sample = df['Fare'].sample(30, replace=True)
    sample_mean = sample.mean()
    sample_means.append(sample_mean)

sample_means = np.array(sample_means)

print("Number of sample means collected:", len(sample_means))


Number of sample means collected: 1000


In [35]:
# Compare the skewness of the original Fare column vs the skewness of the sample means

original_skew = skew(df['Fare'])
sampling_skew = skew(sample_means)

print("Population Fare skewness:", original_skew)
print("Sampling distribution skewness:", sampling_skew)


Population Fare skewness: 4.805731644033497
Sampling distribution skewness: 0.8457727290801509


**Why this matters:** the sampling distribution's skewness is much closer to 0 than the raw
Fare column's skewness. This proves CLT in practice — even skewed raw data produces
roughly-normal sample means once you average enough of them together.


## Day 13 — Confidence Intervals

**Simple idea:** instead of one single average, we give a range we're fairly confident the true
average falls inside.


In [36]:
# Build a 95% confidence interval for the mean Age

n = len(age)
mean_age = age.mean()
sem_age = stats.sem(age)                     # standard error of the mean

t_critical = stats.t.ppf(0.975, df=n - 1)    # 0.975 because we want the middle 95%

margin_of_error = t_critical * sem_age

lower_bound = mean_age - margin_of_error
upper_bound = mean_age + margin_of_error

print("Mean age:", mean_age)
print("95% confidence interval:", lower_bound, "to", upper_bound)


Mean age: 30.25643250688705
95% confidence interval: 28.557576992165636 to 31.955288021608467


In [37]:
# Do the same for Fare

n_fare = len(fare)
mean_fare = fare.mean()
sem_fare = stats.sem(fare)

t_critical_fare = stats.t.ppf(0.975, df=n_fare - 1)
margin_of_error_fare = t_critical_fare * sem_fare

lower_bound_fare = mean_fare - margin_of_error_fare
upper_bound_fare = mean_fare + margin_of_error_fare

print("Mean fare:", mean_fare)
print("95% confidence interval:", lower_bound_fare, "to", upper_bound_fare)


Mean fare: 32.02798675496689
95% confidence interval: 28.809384156665367 to 35.24658935326841


**Why the t-distribution instead of z:** we don't know the true population standard
deviation — we're estimating it from our sample. The t-distribution is wider than the normal
distribution to account for that extra uncertainty.

**Correct interpretation:** we are 95% confident the true population mean falls in that range —
not that 95% of individual passengers fall in that range.


## Day 14 — Covariance

**Simple idea:** covariance tells us the direction of a relationship between two numbers, but
not how strong it is in an easy-to-read way.


In [38]:
# Covariance between Age and Fare

age_fare_data = df[['Age', 'Fare']].dropna()
covariance_matrix = age_fare_data.cov()

print(covariance_matrix)


             Age         Fare
Age   543.628323    37.374819
Fare   37.374819  2763.890980


In [39]:
# Compare covariance to correlation for the same pair

cov_value = covariance_matrix.loc['Age', 'Fare']
correlation_matrix = age_fare_data.corr()
corr_value = correlation_matrix.loc['Age', 'Fare']

print("Covariance (Age, Fare):", cov_value)
print("Correlation (Age, Fare):", corr_value)


Covariance (Age, Fare): 37.3748188829277
Correlation (Age, Fare): 0.03049071472095925


**Why we report correlation, not covariance, as "how strong":** covariance's raw number is
in mixed units (years times pounds) and has no easy scale. Correlation rescales it to always sit
between -1 and 1, which is why it's the number we actually use to describe relationship strength.


## Day 15 — Exploratory Data Analysis (EDA)

**Simple idea:** EDA isn't a new technique — it's running everything from Day 1 to Day 14
side by side across the whole dataset, to see the full picture at once.


In [40]:
# 1. Shape and missing values

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.isna().sum())


Rows: 906
Columns: 12
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            180
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          682
Embarked         2
dtype: int64


In [41]:
# 2. Basic numeric summary

print(df[['Age', 'Fare']].describe())


              Age        Fare
count  726.000000  906.000000
mean    30.256433   32.027987
std     23.315838   49.363069
min    -62.000000    0.000000
25%     20.000000    7.921250
50%     28.000000   14.454200
75%     38.000000   31.000000
max    300.000000  512.329200


In [42]:
# 3. Categorical breakdown

print(df['Sex'].value_counts())
print(df['Pclass'].value_counts())
print(df['Embarked'].value_counts())


Sex
male      585
female    321
Name: count, dtype: int64
Pclass
3    500
1    219
2    187
Name: count, dtype: int64
Embarked
S    654
C    171
Q     79
Name: count, dtype: int64


In [43]:
# 4. Survival rate by group

print(df.groupby('Sex')['Survived'].mean())
print(df.groupby('Pclass')['Survived'].mean())


Sex
female    0.744548
male      0.188034
Name: Survived, dtype: float64
Pclass
1    0.630137
2    0.470588
3    0.246000
Name: Survived, dtype: float64


In [44]:
# 5. Correlation matrix across key numeric columns

correlation_summary = df[['Age', 'Fare', 'Pclass', 'Survived']].corr()
print(correlation_summary)


               Age      Fare    Pclass  Survived
Age       1.000000  0.030491 -0.177353 -0.080280
Fare      0.030491  1.000000 -0.549514  0.255312
Pclass   -0.177353 -0.549514  1.000000 -0.334721
Survived -0.080280  0.255312 -0.334721  1.000000


**Why this final pass matters:** each earlier day tested one relationship at a time with
the correct tool for that data type. This last EDA pass lays every result side by side, so
patterns that reinforce each other become visible — Sex, Pclass, and Fare all point toward the
same conclusion from different angles.


## Saving the Cleaned Data

We save the fully cleaned table to a brand new CSV file. We never overwrite the original messy
file — we always keep the raw data untouched and save cleaned versions separately.


In [45]:
# Save the cleaned DataFrame to a new CSV file
# index=False means we don't save the row numbers as an extra column

df.to_csv('titanic_cleaned.csv', index=False)

print("Cleaned file saved as titanic_cleaned.csv")
print("Shape of cleaned data:", df.shape)


Cleaned file saved as titanic_cleaned.csv
Shape of cleaned data: (906, 12)


## Summary

- Real data must be cleaned before any statistics can be trusted — we fixed whitespace,
  inconsistent case, a hidden currency symbol, and a fake "missing" text marker
- Day 1–5 covered describing one variable at a time: central tendency, spread, and shape
- Day 6–7 covered probability, including flipping a conditional question with Bayes' Theorem
- Day 8 covered measuring relationships between two variables, choosing Pearson or Spearman
  based on whether the data was truly numeric or ordinal
- Day 9–11 covered formally testing whether a difference or relationship is real: t-test for
  numeric-vs-group comparisons, chi-square for category-vs-category comparisons
- Day 12–13 covered the Central Limit Theorem and confidence intervals, which justify using
  normal-based methods even on skewed raw data
- Day 14 covered covariance as the raw ingredient behind correlation
- Day 15 combined everything into one full EDA pass
- The cleaned data was saved to `titanic_cleaned.csv` for future use
